# 7.20 下一個之外，還能一起練習預測什麼？


讀到「雨」，下一個字可能是「停」；若訓練材料的完整句子是「雨停了」，再下一個則是「了」。以前每個位置只交一份下一步的答案。現在想讓模型讀同一段前文時，也為稍遠的未來做準備，可以多交一份「再下一步」的答案。

這種訓練稱為 **Multi-Token Prediction（MTP，多 token 預測）**。它改的是學習目標：在同一個前文位置，要求模型預測不止一個未來的 token。我們希望這些額外練習，能讓共用的特徵包含對後續有用的線索；是否真的改善能力，要看資料、架構與訓練條件。先看目標怎麼改，不把它當成已學會整句話。

本例特別指定每個中文字各是一個 token，另有開始標記〈B〉（即 BOS）與結束標記〈E〉（即 EOS），它們也各佔一個位置。訓練序列是「〈B〉／雨／停／了／〈E〉」，斜線只表示分隔。讀到「雨」的位置，依 [3.6](https://birdhackor.github.io/tiny-perceptron-vlm/3.6.html) 的因果遮罩只能使用〈B〉和雨，不能先讀到停與了。

模型裡負責讀前文、替每個位置算出特徵的主體，也就是 embedding、注意力與 FFN 這些層，下文叫**核心**。把讀到「雨」的那份特徵送到兩條預測分支：**距離 1** 的分支預測往後數第 1 個 token「停」，**距離 2** 的分支預測往後數第 2 個 token「了」，中間隔著一個停。每條分支先用自己的一小段網路處理這份特徵，最後都交給同一個輸出層（[4.6](https://birdhackor.github.io/tiny-perceptron-vlm/4.6.html) 那種把特徵變成每個候選分數的層，也叫 LM head）算出分數。選讀引用的論文把每條分支自己的那一小段網路叫 head；它既不是 [3.7](https://birdhackor.github.io/tiny-perceptron-vlm/3.7.html) 的注意力頭，也不是共用的輸出層。兩條分支共用核心與輸出層，卻各有自己的預測任務。


In [ ]:
# @title 準備本節的工具（首次執行）
from pathlib import Path
import os
import subprocess
import sys

# Colab 會先下載專案；本機 Jupyter 沿用已開啟的 repo。
try:
    import google.colab
except ImportError:
    in_colab = False
else:
    in_colab = True

if in_colab:
    root = Path("/content/tiny-perceptron-vlm")
    if not root.exists():
        subprocess.run(
            ["git", "clone", "--depth", "1", "https://github.com/birdhackor/tiny-perceptron-vlm.git", str(root)],
            env={**os.environ, "GIT_LFS_SKIP_SMUDGE": "1"},
            check=True,
        )
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-e", str(root)], check=True)
    os.chdir(root)

root = Path.cwd().resolve()
while not (root / "pyproject.toml").exists() and root != root.parent:
    root = root.parent
if not (root / "tiny_perceptron").is_dir():
    raise RuntimeError("本機請先依 course/first-steps.md 開啟 repo；免安裝練習可使用本節的 Colab 入口")
sys.path.insert(0, str(root))
import torch

torch.set_num_threads(1)
torch.manual_seed(42)

In [ ]:
# @title 本節圖解
from IPython.display import SVG, display

display(SVG(filename=str(root / "course/figures/p7-7.20-future-targets.svg")))

圖中的「停」與「了」是用來核對的正確答案，不是事先塞給核心的輸入。在這個平行版本中，距離 2 的分支也不先接距離 1 猜出的停；兩條都從相同的已讀前文預測，所以可能各自猜錯，也可能彼此不搭配。

每往後移一個位置，兩條分支的正確答案也跟著移動。下面逐個位置列出讀到的前文，以及兩個距離的目標：


In [ ]:
tokens = ["〈B〉", "雨", "停", "了", "〈E〉"]
for t in range(len(tokens) - 1):
    read = "／".join(tokens[: t + 1])
    near = tokens[t + 1]
    far = tokens[t + 2] if t + 2 < len(tokens) else "（沒有目標）"
    print(f"位置{t} 讀到 {read}｜距離1：{near}｜距離2：{far}")

`range(len(tokens) - 1)` 是位置 0 到 3：最後的〈E〉後面沒有下一項，不必再預測。`tokens[: t + 1]` 是位置 t 能讀的前文，`"／".join(...)` 用斜線把它們接成一段文字，方便顯示。距離 d 的目標就是 `tokens[t + d]`；位置 3 讀到「了」時，往後數兩個已經超出序列，所以距離 2 沒有目標，[7.21](https://birdhackor.github.io/tiny-perceptron-vlm/7.21.html) 會處理這種位置。

訓練時，這四個位置可以一起算，但因果遮罩仍限制每個位置只讀當時的前文。「一次同時算很多個位置」和「同一個位置多預測一個未來答案」，是兩種不同的改動。

練習把 tokens 改成「〈B〉／雨／又／停／了／〈E〉」，每字仍各一個 token。先預測讀到「雨」的位置 1，兩條分支各該學什麼，再執行核對。

<details>
<summary>練習的答案思路</summary>

位置 1 讀到〈B〉／雨，距離 1 的目標是「又」，距離 2 的目標是「停」。距離 2 的意思是往後數第 2 個 token，不是「永遠猜句子的最後一字」，所以這時還輪不到「了」。

</details>

<details>
<summary>選讀：這裡的平行分支對應哪個來源？</summary>

[Gloeckle 等人 2024 年原論文的第 2 節](https://arxiv.org/pdf/2404.19737v1)使用共用的核心與各自獨立的未來分支。原文的分支含 Transformer 層，之後共用同一個輸出層，也就是把特徵映成字表分數的矩陣；本圖只畫任務關係，沒有畫出分支內部的層。原論文的比較還會調整核心與分支的層數，不能把增加分支說成免費增加能力。

只用距離 1 的分支，也能照常**逐 token 生成**（也叫自回歸，autoregressive）：每次只生成一個 token，接回輸入再算下一個，就像 [1.14](https://birdhackor.github.io/tiny-perceptron-vlm/1.14.html)。額外的分支能否拿來加速，另需草稿與驗證程序，見 [16.14](https://birdhackor.github.io/tiny-perceptron-vlm/16.14.html)。

</details>
